# Common Functions definitions and variables

## Install, import Libraries and Global Variables

In [1]:
%%capture --no-display
!pip install dkpro-cassis
!pip install spacy-sentiws
!python -m spacy download de_core_news_sm
!pip install fuzzywuzzy

In [2]:
#Libraries import.
from cassis import * #creation and manipulation of annotated documents: dkpro-cassis
from pathlib import * ##Not used
from collections import Counter
#Load annotation data into the notebook
from google.colab import drive
from google.colab import files
import os # Define the files to be deleted and modified in google Colab
import csv #Save to CSV after calcuations
import zipfile #For zip files
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz #for naming mistakes similarity scores.
from collections import defaultdict
import plotly.express as px

from datetime import datetime
import pytz
import shutil


/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:
tz_Berlin = pytz.timezone('Europe/Berlin') #Berlin time when saving date in the filename of xlsx
DEBUG = False #Used if need output from from methods.
PROCESS_SINGLE_AUTHOR = False # Used testing purpose. When True then have to upload zip file of single author.
THRESHOLD = 90 #fuzzywuzzy for naming mistakes similarity scores.

## Fuction to delete all files

In [4]:
def action_delete_all_files(directory):
  '''
  Delete all files.
  '''
  try:
      # Check if the directory exists
      if os.path.exists(directory) and os.path.isdir(directory):
          # Iterate over all files and directories within the directory
          for filename in os.listdir(directory):
              file_path = os.path.join(directory, filename)

              # Check if it's a file and delete it
              if os.path.isfile(file_path) or os.path.islink(file_path):
                  os.remove(file_path)
                  print(f"Deleted file: {file_path}")
              # Check if it's a directory (optional) and skip or delete recursively
              elif os.path.isdir(file_path):
                  shutil.rmtree(file_path)  # Use this if you want to delete subdirectories
                  print(f"Deleted directory: {file_path}")
      else:
          print("Directory does not exist or is not a directory.")
  except Exception as e:
      print(f"Error while deleting files: {e}")

## csv to xlsx with filename_date_time for required columns

In [5]:
def save_xlsx_with_date_time(filename, columns=None,save_csv_without_date_time=False):
    '''
    Takes a CSV and saves it with the required columns to an XLSX OR CSV file.
    If columns are not specified, all columns are saved.
    '''

    # Load the CSV file into a DataFrame
    df = pd.read_csv(filename, encoding='utf-8-sig')

    # If no columns are specified, use all columns
    if columns is None:
        df_filtered = df
    else:
        df_filtered = df[columns]

    # Sort the DataFrame by 'Document_title' column if it exists (just as a safety ameasure)
    if 'Document_title' in df_filtered.columns:
        df_sorted = df_filtered.sort_values(by='Document_title')
    else:
        df_sorted = df_filtered  # No sorting if 'Document_title' does not exist

    # Get the current date and time in the format YYYYMMDD
    current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')


    if(save_csv_without_date_time):
      new_file_name = new_file_name = f'{filename.split(".")[0]}.xlsx'
      df_sorted.to_csv(new_file_name, encoding='utf-8-sig', index=False)
    else:
      # Define the new Excel file name with the current date and time
      new_file_name = f'{filename.split(".")[0]}_{current_time}.xlsx'

      # Save the sorted DataFrame to a new Excel file
      df_sorted.to_excel(new_file_name, index=False)

    print(f"File saved as: {new_file_name}")

## Combine two csv on a key

In [6]:
def combine_csv(filenames, outputfilename, key='Document_title'):
    """
    Combines multiple CSV files on a common key and saves the output to a new file.

    Parameters:
    filenames (list of str): List of CSV filenames to combine.
    key (str): The column name to merge the CSV files on.
    outputfilename (str): The name of the output CSV file.
    """
    # Initialize an empty DataFrame to hold the combined data
    combined_df = pd.DataFrame()

    for filename in filenames:
        if os.path.exists(filename):
            # Read the current CSV file
            df = pd.read_csv(filename)

            # If combined_df is empty, initialize it with the current dataframe
            if combined_df.empty:
                combined_df = df
            else:
                # Merge the current CSV with the existing combined dataframe on the specified key
                combined_df = pd.merge(combined_df, df, on=key, how='inner')
        else:
            print(f"File {filename} does not exist.")

    # Save the combined dataframe to the output file
    combined_df.to_csv(outputfilename, index=False, encoding='utf-8-sig')
    print(f"Combined CSV saved to {outputfilename}")


# Text Analysis

## Function definitions

### Function to ensure that None values are converted to strings

In [7]:
# Function to ensure that None values are converted to strings
def safe_str(obj):
    return str(obj) if obj is not None else 'NONE'

Function to upload just single author .zip

In [8]:
#Upload zip file intcontaining CAS XMI and XML
def upload_next_author_zip():

  #First delete files of previous author, if any.
  # Define the files to be deleted
  files_to_delete = ['TypeSystem.xml', 'CURATION_USER.xmi']

  # Path to the folder where the files are located
  content_folder_path = '/content/'

  # Delete the specified files if they exist
  for file_name in files_to_delete:
      file_path = os.path.join(content_folder_path, file_name)
      if os.path.isfile(file_path):
          os.remove(file_path)
          print(f"Deleted {file_path}")
      else:
          print(f"File {file_path} does not exist.")

  print(f"Deletion process completed for {files_to_delete} from path: {content_folder_path}.")
  print("\n")

  #Upload the ZIP file
  uploaded = files.upload()

  # Specify the extraction path
  content_folder_path = '/content/'

  # Extract the uploaded ZIP file to the specified folder
  for filename in uploaded.keys():
      # Ensure it's a zip file
      if filename.endswith('.zip'):
          with zipfile.ZipFile(filename, 'r') as zip_ref:
              zip_ref.extractall(content_folder_path)
              print(f'Extracted all files from {filename} to {content_folder_path}')
      else:
          print(f'{filename} is not a zip file')

  # Delete the ZIP file after extraction
  os.remove(filename)
  print(f'Deleted the ZIP file: {filename}')

### Functions of uploading zip file for all authors and extracting zip files and retun the list of folders.

In [9]:
#Extract files from from zip. for all authors "webanno*****export_curated_documents.zip"
def extract_nested_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as main_zip:
        # Extract the main zip
        main_zip.extractall(extract_to)

        # Iterate through each folder in the curation directory
        for folder_name in main_zip.namelist():
            if folder_name.endswith('.zip') and 'curation' in folder_name:
                nested_zip_path = os.path.join(extract_to, folder_name)

                # Extract the nested zip files
                with zipfile.ZipFile(nested_zip_path, 'r') as nested_zip:
                    nested_zip.extractall(os.path.dirname(nested_zip_path))

#returns the folder names as list in './curation'.
def get_folder_names(curation_path='./curation'):
    folder_names = [folder_name for folder_name in os.listdir(curation_path)
                    if os.path.isdir(os.path.join(curation_path, folder_name))]
    return folder_names



### Function for resetting variables to default value and default variables.

In [10]:
#Reset the values to default for variable: "annotation_overview_table"
def reset_annotation_overview_table(debug):
  global annotation_overview_table
  for key, value in annotation_overview_table.items():
      if isinstance(value, int):
          annotation_overview_table[key] = 0
      elif isinstance(value, str):
          annotation_overview_table[key] = '-'
  if debug:
    print(annotation_overview_table)
  return annotation_overview_table

In [11]:
#All entries of the annotaion overview scheme to be calculated
annotation_overview_table = {
                             'Document_title' : '',
                             'Textlänge_token' : 0, #Text length (tokens)
                             'Textlänge_sent' : 0 , #Text length (sentences)
                             'Entität_gesamt' : 0, 	#Entity (total)
                             'Entität_Ding'  : 0, #Entity (thing)
                             'Entität_Figur' : 0, #Entity (character)
                             'Entität_Figurengruppe' : 0, 	#Entity (character group)
                             'Entität_Raum' : 0, 	#Entity (space)
                             'Entität_Bereich' : 0, #Entity (area)
                             'Entität_Grenze'	 : 0, #Entity (boundary)
                             'Entität_Zeitabschnitt' : 0, 	#Entity (time period)
                             'Entität_Sonstige' : 0, #Entity (other)

                              'Wertung_gesamt' : 0, #Evaluation (total)
                              'Erzählerwertung' : 0, #Narrator evaluation
                             'Figurenwertung' : 0, #Character evaluation
                             'Wertung_Kompositionsebene' : 0,  #Evaluation (composition level)
                             'positive_Wertung' : 0,  #Positive evaluation
                             'negative_Wertung' : 0,  #Negative evaluation
                             'positive_Wertung_mit_Ironie': 0, #Positive evaluation with ironie
                             'negative_Wertung_mit_Ironie': 0, #negative evaluation with ironie

                             'Wertung_moralisch' : 0,  #Moral evaluation
                             'Wertung_eudämonistisch' : 0,  #Eudaimonistic evaluation
                             'Wertung_sozialer_Status' : 0,  #Social status evaluation
                             'Wertung_ästhetisch' : 0,  #Aesthetic evaluation
                             'Wertung_sonstige' : 0,  #Other evaluation


                             'Wertung_eudämonistisch_figure' : 0, #eudämonistisch counts if entity is figure
                             'Wertung_eudämonistisch_not_figure' : 0,



                            'NP_Wertung' : 0,  #NP evaluation (##Not implemented)
                            'Teilsatzwertung' : 0,  #Clause evaluation (##Not implemented)

                             'Codierung_gesamt' : 0,  #Coding (overall)
                             'Codierung_tief_oberflächlich' : 0,  #Coding (deep vs. superficial)
                             'Codierung_natürlich_kulturell' : 0,  #Coding (natural vs. cultural)
                             'Codierung_traditionell_modern' : 0,  #Coding (traditional vs. modern)
                             'Codierung_harmonisch_disharmonisch' : 0,  #Coding (harmonious vs. disharmonious)
                             'Codierung_Gesund_krank' : 0,  #Coding (healthy vs. sick opposition)

                             'Opposition_gesamt' : 0,  #Opposition (overall)
                             'Wertopposition' : 0,  #Value opposition
                             'Wertopposition_Oppositionspaar': '', #Value opposition List
                             'Traditionell-modern-Opposition' : 0,  #Traditional vs. modern opposition
                             'Harmonisch-disharmonisch-Opposition' : 0,  #Harmonious vs. disharmonious opposition
                             'Natürlich-kulturell-Opposition' : 0,  #Natural vs. cultural opposition
                             'Gesund-krank-Opposition' : 0,  #Healthy vs. sick opposition
                             'Tiefgründig-oberflächlich-Opposition' : 0,  #Profound vs. superficial opposition
                             'Sonstige_Merkmalsopposition' : 0,  #Other feature opposition
                             'Merkmalsopposition_Oppositionspaar': '', #Other feature opposition List
                             'Zugehörigkeit_zu' : 0,  #Belonging to
                             'Element_von' : 0,  #Element of
                             #'Kritikhaltige_Reflexion':0, #Kritikhaltige Reflexion




                             'Span_FIGUR' : 0,
                             'Span_LISTE_FIGUR' : '',

                             'Span_RAUM' : 0,
                             'Span_LISTE_RAUM' : '',

                             'Span_DING' : 0,
                             'Span_LISTE_DING' : '',

                             'Span_SONSTIGE' : 0,
                             'Span_LISTE_SONSTIGE' : '',

                             'Span_FIGURENGRUPPE' : 0,
                             'Span_LISTE_FIGURENGRUPPE' : '',

                             'Span_BEREICH' : 0,
                             'Span_LISTE_BEREICH' : '',

                             'Span_GRENZE' : 0,
                             'Span_LISTE_GRENZE' : '',

                             'Span_ABSTRAKTUM' : 0,
                             'Span_LISTE_ABSTRAKTUM' : '',

                             'Span_ZEITABSCHNITT' : 0,
                             'Span_LISTE_ZEITABSCHNITT' : '',

                             'Span_ORGANISATION' : 0,
                             'Span_LISTE_ORGANISATION' : '',

                             'Span_NONE' : 0, #In case there is no label (in case of author/Narrator)
                             'Span_LISTE_NONE' : '', #In case there is no label (in case of author/Narrator)

                             'Span_entities_with_Referenz_Wikidata':0,
                             'Span_LISTE_entities_with_Referenz_Wikidata' : '-',
                              'total_list_naming_mistakes':0,
                              'list_naming_mistakes': '',

                              'Erzählerwechsel' : 0,   #Five new categories added dated:17-09-2024
                              'Kritikhaltige_Reflexion' : 0,
                              'TriggerCodierung' : 0,
                              'TriggerWertung' : 0,
                              'TriggerMerkmalsopposition' : 0,
                              'Overlapping_Oppositions' : 0,
                             'Span_LISTE_Overlapping_Oppositions' : '-', #overlapping oppositions Added: 28-09-2024
                             #'Span_LISTE_Overlapping_Oppositions_Potential_ERRORS': '-',
                             'evaluative_difference_of_contrastive_entities': 0, #Added: 05-19-2024
                             'codierung_with_trigger_explicit':0, #Added: 31-10-2024
                             'codierung_without_trigger_implicit':0,
                             'wertung_with_trigger_explicit':0,
                             'wertung_without_trigger_implicit':0,
                             'ratio_explicit_implicit_codierung':0,
                             'ratio_explicit_implicit_wertung':0
                            }

### Function to check file availability

In [12]:
#Checks if file is available if not then uploads it.
def check_for_file(file_name):
    # Define the file to be deleted
    file_namex = file_name

    # Path to the folder where the file is located
    content_folder_path = '/content/'
    file_path = os.path.join(content_folder_path, file_namex)

    # Check if the file exists
    if not os.path.isfile(file_path):
        # If the file does not exist, ask for upload
        print(f"File {file_path} does not exist. Upload {file_name} may be generated from last step")
        file_to_open = files.upload()
    else:
      print(f"{file_namex} Already uploaded.\n")

### Function to for saving to CSV file

In [14]:
#Updated Save to CSV now it sorts by Document_title while saving.
def save_to_csv(annotation_overview_table,file_name='01_output.csv'):
    # Define the CSV file name
    csv_file = file_name

    # Check if the CSV file already exists
    file_exists = os.path.isfile(csv_file)

    # If the file exists, read it into a pandas DataFrame
    if file_exists:
        df = pd.read_csv(csv_file, encoding='utf-8-sig')
    else:
        df = pd.DataFrame()

    # Convert the annotation_overview_table dictionary to a DataFrame
    new_row_df = pd.DataFrame([annotation_overview_table])

    # Append the new row to the existing DataFrame
    df = pd.concat([df, new_row_df], ignore_index=True)

    # Sort the DataFrame by 'Document_title' (or any other column you want)
    if 'Document_title' in df.columns:
        df = df.sort_values(by='Document_title')

    # Save the sorted DataFrame to the CSV file
    df.to_csv(csv_file, mode='w', index=False, encoding='utf-8-sig')

    print(f"Data appended and sorted in {csv_file}")


### Funtions to count Tokens and Sentences

In [15]:
#Count Analysis of Tokens
#Return Token count from the document
def token_info(doc, debug):
  #Textlänge_token
  #Text length (tokens)
  token_count = sum(1 for count in doc.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'))
  if debug:
    print("#################### Textlänge_token (Number of (tokens)) #####################################")
    print(f'Token_Count: {token_count}')
    print("\n")
  return token_count


#Count Analysis of Sentences
def sen_info(doc,debug):
  #Textlänge_sent
  #Text length (sentences)
  sentence_count = sum(1 for count in doc.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence'))
  if debug:
    print("#################### Textlänge_sent (Number of (sentences)) #####################################")
    print(f'Sentence_Count: {sentence_count}')
    print("\n")

  #Calculated values from this cell.
  annotation_overview_table['Textlänge_sent'] = sentence_count #Text length (sentences)

### Funtion to count Entities(Entität)

In [16]:
#Count Analysis of Entities
def entities_info(doc, _doc_title,debug):
  #Entität_gesamt	Entität_Ding	Entität_Figur	Entität_Figurengruppe	Entität_Raum	Entität_Bereich	Entität_Grenze	Entität_Zeitabschnitt	Entität_Sonstige
  #Entity (overall)	Entity (thing)	Entity (character)	Entity (character group)	Entity (space)	Entity (area)	Entity (boundary)	Entity (time period)	Entity (other)

  # Initialize counts
  Entities_counts = {'DING' : 0,
                    'FIGUR': 0,
                    'FIGURENGRUPPE': 0,
                    'RAUM': 0,
                    'BEREICH' : 0,
                    'GRENZE': 0,
                    'ZEITABSCHNITT': 0,
                    'SONSTIGE': 0
  }
  entities_overlap = {}  # Use a dict to map token to its (begin, end) span 04-05-25

  #Get the entity label and count if in the list of entity type.

  for idx, token in enumerate(doc.select('custom.Span')):
      if token.label in Entities_counts:
        Entities_counts[token.label] += 1
        entities_overlap[idx] = (token, token.label, token.begin, token.end)

  # Step 2: Filter out tokens with no overlap
  to_remove = []
  for idx1, (token1, _, begin1, end1) in entities_overlap.items():
      has_overlap = False
      for idx2, (token2, _, begin2, end2) in entities_overlap.items():
          if idx1 != idx2 and begin1 <= end2 and begin2 <= end1:
              has_overlap = True
              break
      if not has_overlap:
          to_remove.append(idx1)

  for idx in to_remove:
      del entities_overlap[idx]


  # Step 3: Save to txt

  with open("entities_overlap.txt", 'a', encoding='utf-8') as f:
      f.write(f"============{_doc_title}===========\n")
      for idx, (token, label, start, end) in entities_overlap.items():
          f.write(f"[{idx}] [covered_text:{token.get_covered_text()}] [token:{token}]\n")

  if debug:
    print("#################### Entität (Entity Counts) #####################################")
    print(f'Gesamt: {sum(Entities_counts.values())}::',Entities_counts)
    print("\n")


  #Calculated values from this cell.
  annotation_overview_table['Entität_gesamt'] = sum(Entities_counts.values()) #Entity (total)
  # Create a mapping of Entities_counts keys to annotation_overview_table keys
  mapping = {'DING' : 'Entität_Ding',
                    'FIGUR': 'Entität_Figur',
                    'FIGURENGRUPPE': 'Entität_Figurengruppe',
                    'RAUM': 'Entität_Raum',
                    'BEREICH' : 'Entität_Bereich',
                    'GRENZE': 'Entität_Grenze',
                    'ZEITABSCHNITT': 'Entität_Zeitabschnitt',
                    'SONSTIGE': 'Entität_Sonstige'
  }

  # Update annotation_overview_table_key with the values from Entities_counts based on the mapping
  for Entities_counts_key, annotation_overview_table_key in mapping.items():
    if Entities_counts_key in Entities_counts:
        annotation_overview_table[annotation_overview_table_key] = Entities_counts[Entities_counts_key]

### Funtion to count Ratings/Evaluations(Wertung)

In [ ]:
#19-12-2024
def info_Wertung_n_and_tokens_desctiptive_stats(doc_CAS_xmi, debug=False):

  if(debug):
    print("=====info_Wertung_n_and_tokens_desctiptive_stats=====")

  doc_CAS_xmix = doc_CAS_xmi
  n=0
  count_dependent_tokens = 0
  for token in doc_CAS_xmix.select('webanno.custom.Wertung'):
    n+=1
    #Count no of tokens with in the encodings
    for doc_tokens in doc_CAS_xmix.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'):
      if(doc_tokens.begin)>=token.begin and (doc_tokens.end)<=token.end:
        count_dependent_tokens+=1

  return n,count_dependent_tokens


#Count Analysis of Ratings/Evaluation
def evaluation_info(doc,debug): #Exlcluding Positive and negative count as they are calulcated in Evaluation_count_positive_negative():
  debug=True
  #Wertung_gesamt	Erzählerwertung	Figurenwertung	Wertung_Kompositionsebene	 	Wertung_moralisch	Wertung_eudämonistisch	Wertung_sozialer_Status	Wertung_ästhetisch	Wertung_sonstige
  #Evaluation (total)	Narrator evaluation	Character evaluation	Evaluation (composition level) Moral evaluation	Eudaimonistic evaluation	Social status evaluation	Aesthetic evaluation	Other evaluation



  # Initialize counts
  evaluations_counts = {'Erzählerwertung' : 0 , 'Figurenwertung' : 0 ,	'Kompositionsebene' : 0 ,
                    #'positive_Wertung' : 0 ,	'negative_Wertung' : 0 , #calculated in evaluation_count_positive_negative():
                    'moralisch' : 0 ,	'eudämonistisch' : 0 ,
                    'sozialer Status' : 0 ,	'ästhetisch' : 0 ,	'sonstige' : 0
  }

  #Get the label and count if in the list of Evaluation types.
  for token in doc.select('webanno.custom.Wertung'):
    if token.Wertungshinsicht in evaluations_counts:
      evaluations_counts[token.Wertungshinsicht] += 1
    elif token.Wertungshinsicht =='nicht spezifiziert': #Added count of nicht spezifiziert to sonstige
      evaluations_counts['sonstige'] += 1
    if token.Wertung_auf_Kompositionsebene == 'Kompositionsebene':
      evaluations_counts['Kompositionsebene'] += 1

  #finding Erzählerwertung and Figurenwertung
  Figurewertung = 0
  Erzahelrwertung = 0
  Wertung_gesamt=0

  for segment in doc.select('webanno.custom.Wertung'):
      Wertung_gesamt+=1
      flagx=True
      if segment.TEST:
        for e in segment.TEST.elements:
          if e.role == "Wertender":#FigureWertung.
            flagx=False
          if e.role not in ["Wertender" ,"gewertete Entität"]:
            print("============== e.role not in [Wertender ,gewertete Entität] == :", e.target.get_covered_text(), " Role: ", e.role)

      if flagx:
        Erzahelrwertung+=1
      else:
        Figurewertung+=1

  evaluations_counts['Figurenwertung'] = Figurewertung
  evaluations_counts['Erzählerwertung'] = Erzahelrwertung

  #TODO: 09-01-2025 total evaluations = gesamt = FigureWertung.+ #Erzahelrwertung.
  Wertung_gesamt=evaluations_counts['Erzählerwertung']+evaluations_counts['Figurenwertung']#+evaluations_counts['Kompositionsebene']


  if debug:
    print("####################  Wertung (Analysis of Ratings ) : Excluding positive_Wertung	negative_Wertung #####################################")
    print('\n')
    print("Figurenwertung",Figurewertung)
    print("Erzählerwertung",Erzahelrwertung)
    print("\n")

  annotation_overview_table['Wertung_gesamt'] = Wertung_gesamt #Ratings/Evaluation (total)

  # Create a mapping of evaluations_counts keys to annotation_overview_table keys
  mapping = {'Erzählerwertung' : 'Erzählerwertung' ,
                    'Figurenwertung' : 'Figurenwertung' ,	'Kompositionsebene' : 'Wertung_Kompositionsebene' ,
                    #'positive_Wertung' : 'positive_Wertung' ,	'negative_Wertung' : 'negative_Wertung' ,
                    'moralisch' : 'Wertung_moralisch' ,	'eudämonistisch' : 'Wertung_eudämonistisch',
                    'sozialer Status' : 'Wertung_sozialer_Status' ,	'ästhetisch' : 'Wertung_ästhetisch' ,	'sonstige' : 'Wertung_sonstige'
  }

  # Update annotation_overview_table_key with the values from coding_counts based on the mapping
  for coding_counts_key, annotation_overview_table_key in mapping.items():
    if coding_counts_key in evaluations_counts:
        annotation_overview_table[annotation_overview_table_key] = evaluations_counts[coding_counts_key]


### Function for Wertung_eudämonistisch and Wertung_eudämonistisch__figure counts

In [18]:
def eudämonistisch_fig_notfig_counts(typesystem_xml):
  with open(typesystem_path, 'rb') as f:
    typesystem_xml = load_typesystem(f)

  with open(cas_file_path, 'rb') as f:
    doc_CAS_xmi = load_cas_from_xmi(f, typesystem=typesystem_xml) # cassis requires xml 1.0  not 1.1! somehow it seems to work nevertheless?

  eudämonistisch_total = 0
  eudämonistisch__figure = 0

  # Loop through all 'webanno.custom.Wertung' annotations
  for wertung_token in doc_CAS_xmi.select('webanno.custom.Wertung'):
      if wertung_token.Wertungshinsicht == 'eudämonistisch':
          eudämonistisch_total += 1

          # Now check if there's a corresponding 'custom.Span' with the label "FIGUR"
          for span_token in doc_CAS_xmi.select('custom.Span'):
              # Check if the spans overlap or are exactly the same
              if span_token.label == "FIGUR" and (
                  span_token.begin == wertung_token.begin and span_token.end == wertung_token.end
              ):
                  eudämonistisch__figure += 1
                  break  # Assuming one match per wertung_token is enough
  eudämonistisch_not_figure= eudämonistisch_total - eudämonistisch__figure
  return (eudämonistisch__figure,eudämonistisch_not_figure)


### Funtion to count Positive, nagative Ratings/Evaluations(Wertung), positive_Wertung_mit_Ironie and nagative_Wertung_mit_Ironie

In [ ]:
# Wertungen ausgeben / Issue ratings
#Erzählerwertung	Figurenwertung	Wertung_Kompositionsebene	positive_Wertung	negative_Wertung
#Narrator evaluation(default)	Character evaluation	Evaluation (composition level)author	Positive evaluation	Negative evaluation
#Count Analysis of Ratings
def evaluations_count_positive_negative(doc,debug):

  # Initialize the dictionary
  dict_positive_entities = {}
  dict_negative_entities = {}
  dict_positive_entities_nameless  = {}
  dict_negative_entities_nameless  = {}
  PIronie=0 #positive_Wertung_mit_Ironie
  NIronie=0 #negative_Wertung_mit_Ironie

  for segment in doc.select('webanno.custom.Wertung'): ##webanno.custom.rating
      Erzählerwertung = True  ##Narrator Rating = TRUE
      try:
          for e in segment.TEST.elements: # and why does this seem to work here but not above?
              #print(e)
              pass
      except:
          #print("Segment ohne gew. Objekt: Segment without selected object") ##Segment without selected object
          continue

      for e in segment.TEST.elements:
          #print(e.role, ": ", e.target.get_covered_text(), ", Typ: ", e.target.label, ", Name: ", e.target.Name, ", Polarität: ", segment.Polaritt)

          # create lists of positive entities
          if Erzählerwertung == True and e.role == "gewertete Entität":
            if (segment.Polaritt == "sehr positiv" or segment.Polaritt == "positiv"):
              if(segment.Ironie):
                PIronie+=1
              if e.target.Name == None:
                if e.target.get_covered_text() not in dict_positive_entities_nameless:
                  dict_positive_entities_nameless[e.target.get_covered_text()] = 1
                else:
                  dict_positive_entities_nameless[e.target.get_covered_text()] += 1

              elif e.target.Name not in dict_positive_entities:
                  dict_positive_entities[e.target.Name] = 1
              else:
                  dict_positive_entities[e.target.Name] += 1


          # create lists of negative entities
          if Erzählerwertung == True and e.role == "gewertete Entität":
            if(segment.Polaritt == "sehr negativ" or segment.Polaritt == "negativ" ):
              if(segment.Ironie):
                NIronie+=1
              if e.target.Name == None:
                if e.target.get_covered_text() not in dict_negative_entities_nameless:
                  dict_negative_entities_nameless[e.target.get_covered_text()] = 1
                else:
                  dict_negative_entities_nameless[e.target.get_covered_text()] += 1
              elif e.target.Name not in dict_negative_entities:
                  dict_negative_entities[e.target.Name] = 1
              else:
                  dict_negative_entities[e.target.Name] += 1


  if debug:

    print("####################  Wertung (Analysis of Ratings ) : positive_Wertung	negative_Wertung #####################################")

    print("\n")

    print("Total: dict_positive_entities= ",sum(dict_positive_entities.values()), dict_positive_entities)
    print("Total: dict_positive_entities_nameless= ",sum(dict_positive_entities_nameless.values()), dict_positive_entities_nameless)

    print("\n")

    print("Total: dict_negative_entities= ",sum(dict_negative_entities.values()), dict_negative_entities)
    print("Total: dict_negative_entities_nameless= ",sum(dict_negative_entities_nameless.values()), dict_negative_entities_nameless)

    print("\n")

    #Calculated values from this cell.
  annotation_overview_table['positive_Wertung'] = sum(dict_positive_entities.values()) + sum(dict_positive_entities_nameless.values()) #Positive evaluation
  annotation_overview_table['negative_Wertung'] = sum(dict_negative_entities.values()) + sum(dict_negative_entities_nameless.values()) #Negative evaluation


  annotation_overview_table['positive_Wertung_mit_Ironie']=PIronie
  annotation_overview_table['negative_Wertung_mit_Ironie']=NIronie


### Funtion to count codings(Codierung)

In [20]:
#Count Analysis of Coding
def coding_info(doc,debug):

  #Codierung_gesamt	Codierung_tief_oberflächlich	Codierung_natürlich_kulturell	Codierung_traditionell_modern	Codierung_harmonisch_disharmonisch	Codierung_Gesund_krank_Opposition
  #Coding (Total)	Coding (deep vs. superficial)	Coding (natural vs. cultural)	Coding (traditional vs. modern)	Coding (harmonious vs. disharmonious)	Coding (healthy vs. sick opposition)


  # Initialize counts
  coding_counts = {'TIEF-OBERFLÄCHLICH' : 0,
                  'NATÜRLICH-KULTURELL' : 0,
                  'TRADITIONELL-MODERN' : 0,
                  'HARMONISCH-DISHARMONISCH' : 0,
                  'GESUND-KRANK' : 0
  }
  #Get the label and count if in the list of coding type.
  for token in doc.select('webanno.custom.AltNeuCodierung'):
    if token.Art.upper() in coding_counts:
      coding_counts[token.Art.upper()] += 1

  if debug:
    print("####################  Codierung (Analysis of Coding ) #####################################")
    print(f'Codierung_gesamt: {sum(coding_counts.values())}::',coding_counts)
    print("\n")




  #Calculated values from this cell.
  annotation_overview_table['Codierung_gesamt'] = sum(coding_counts.values()) #Coding (total)
  # Create a mapping of coding_counts keys to annotation_overview_table keys
  mapping = {'TIEF-OBERFLÄCHLICH' : 'Codierung_tief_oberflächlich',
                    'NATÜRLICH-KULTURELL': 'Codierung_natürlich_kulturell',
                    'TRADITIONELL-MODERN': 'Codierung_traditionell_modern',
                    'HARMONISCH-DISHARMONISCH': 'Codierung_harmonisch_disharmonisch',
                    'GESUND-KRANK' : 'Codierung_Gesund_krank'
  }

  # Update annotation_overview_table_key with the values from coding_counts based on the mapping
  for coding_counts_key, annotation_overview_table_key in mapping.items():
    if coding_counts_key in coding_counts:
        annotation_overview_table[annotation_overview_table_key] = coding_counts[coding_counts_key]

### Funtion to count Opposition

In [ ]:
#Count Analysis of Opposition
def opposition_info(doc,debug):

  #Opposition_gesamt	Wertopposition	Traditionell-modern-Opposition	Harmonisch-disharmonisch-Opposition	Natürlich-kulturell-Opposition	Gesund-krank-Opposition	Tiefgründig-oberflächlich-Opposition	Sonstige_Merkmalsopposition
  #Opposition (total)	Value opposition	Traditional vs. modern opposition	Harmonious vs. disharmonious opposition	Natural vs. cultural opposition	Healthy vs. sick opposition	Profound vs. superficial opposition	Other feature opposition

  # for list of Wertopposition_Oppositionspaar and Merkmalsopposition_Oppositionspaar
  opposition_list = {
      'Wertopposition_Oppositionspaar': [], 'Merkmalsopposition_Oppositionspaar': []
  }

  # Initialize counts
  opposition_counts = {'WERTOPPOSITION' : 0,	'TRADITIONELL-MODERN-OPPOSITION' : 0,	'HARMONISCH-DISHARMONISCH-OPPOSITION'	 : 0,
                    'NATÜRLICH-KULTURELL-OPPOSITION' : 0, 'GESUND-KRANK-OPPOSITION' : 0,	'TIEF-OBERFLÄCHLICH-OPPOSITION' : 0, 'SONSTIGE MERKMALSOPPOSITION'  : 0
  }


  #Get the label and count if in the list of Opposition type.
  for token in doc.select('custom.Relation'):

    if token.label.upper()=='WERTOPPOSITION':

      if(token.Wertungshinsicht):
        opposition_list['Wertopposition_Oppositionspaar'].append(safe_str(token.Wertungshinsicht))
      else:
          opposition_list['Wertopposition_Oppositionspaar'].append(safe_str(token.Merkmalsopposition))

    elif token.label.upper()=='SONSTIGE MERKMALSOPPOSITION':
      opposition_list['Merkmalsopposition_Oppositionspaar'].append(safe_str(token.Merkmalsopposition))

    if token.label.upper() in opposition_counts:
      opposition_counts[token.label.upper()] += 1

  if debug:

    print(opposition_list)
  # Update annotation overview table
  for label, names in opposition_list.items():
      if len(names) > 0:
        annotation_overview_table[label] = ";".join(names) + ";"  # Concatenate names


  if debug:
    print("####################  Opposition (Analysis of Opposition ) #####################################")
    print(f'Opposition_gesamt: {sum(opposition_counts.values())}::',opposition_counts)
    print("\n")


  #Calculated values from this cell.
  annotation_overview_table['Opposition_gesamt'] = sum(opposition_counts.values()) #Opposition (total)
  # Create a mapping of coding_counts keys to annotation_overview_table keys
  mapping = {'WERTOPPOSITION' : 'Wertopposition',	'TRADITIONELL-MODERN-OPPOSITION' : 'Traditionell-modern-Opposition',	'HARMONISCH-DISHARMONISCH-OPPOSITION'	 : 'Harmonisch-disharmonisch-Opposition',
                    'NATÜRLICH-KULTURELL-OPPOSITION' : 'Natürlich-kulturell-Opposition',
             'GESUND-KRANK-OPPOSITION' : 'Gesund-krank-Opposition',	'TIEF-OBERFLÄCHLICH-OPPOSITION' : 'Tiefgründig-oberflächlich-Opposition', 'SONSTIGE MERKMALSOPPOSITION'  : 'Sonstige_Merkmalsopposition'
  }

  # Update annotation_overview_table_key with the values from coding_counts based on the mapping
  for opposition_counts_key, annotation_overview_table_key in mapping.items():
    if opposition_counts_key in opposition_counts:
        annotation_overview_table[annotation_overview_table_key] = opposition_counts[opposition_counts_key]


### Funtion to count Zugehörigkeit_zu and Element_von

In [ ]:
#Count Analysis of group of characters/family
def character_family_info(doc,debug):

  #Zugehörigkeit_zu, Element_von

  # Initialize counts
  character_family_counts = {'ZUGEHÖRIGKEIT' : 0,	'ELEMENT VON' : 0  }

  #Get the label and count if in the list.
  for token in doc.select('custom.Relation'):
    if token.label.upper() in character_family_counts:
      character_family_counts[token.label.upper()] += 1

  if debug:
    print("####################  Zugehörigkeit_zu, Element_von (Analysis of group of characters/family ) #####################################")
    print(character_family_counts)
    print("\n")


  # Create a mapping of coding_counts keys to annotation_overview_table keys
  mapping = {'ZUGEHÖRIGKEIT' : 'Zugehörigkeit_zu',	'ELEMENT VON' : 'Element_von'  }

  # Update annotation_overview_table_key with the values from coding_counts based on the mapping
  for character_family_counts_key, annotation_overview_table_key in mapping.items():
    if character_family_counts_key in character_family_counts:
        annotation_overview_table[annotation_overview_table_key] = character_family_counts[character_family_counts_key]


### Funtion to get metadata from the xmi file

In [ ]:
#Gets Title from Metadata.
#Requires: CAS_XMI file
#Retuns: document_title from metadata
def document_metadata_info(typesystem_xml,doc_CAS_xmi,debug):
  # Get the DocumentMetaData type from the type system
  DocumentMetaData = typesystem_xml.get_type('de.tudarmstadt.ukp.dkpro.core.api.metadata.type.DocumentMetaData')
  # Retrieve the DocumentMetaData from the CAS Doc
  document_metadata_list = doc_CAS_xmi.select(DocumentMetaData)

  # Metadata list
  if document_metadata_list:
      document_metadata = document_metadata_list[0] # Access the first (and only) item in the list
      document_title = document_metadata.documentTitle  #Document Title

 #Calculated values from this cell.
  annotation_overview_table['Document_title'] = document_title
  if debug:
    print("#################### DocumentMetaData #####################################")
    print(document_metadata_list)
    print(annotation_overview_table)

  return document_title

### Function to count Kritikhaltige_Reflexion

In [ ]:
#To count Kritikhaltige Reflexion implementation.
def Kritikhaltige_Reflexion_defined(doc):
  Kritikhaltige_Reflexion_counts=0
  #Check for Kritikhaltige Reflexion
  for annotation in doc.select('webanno.custom.Sonstiges'):
    #if(annotation.uiName) == 'Kritikhaltige Reflexion':
    Kritikhaltige_Reflexion_counts+=1
    #print("Kritikhaltige_Reflexion_counts",Kritikhaltige_Reflexion_counts,annotation.get_covered_text(), annotation.value)
  annotation_overview_table['Kritikhaltige_Reflexion']=Kritikhaltige_Reflexion_counts

### Extra Funtion to view implemented properties in xml

In [25]:
#To view implemented properties in xml. (Not essential part of code, for debugging purposes)
def view_xml_properties(typesystem_xml,debug):
  if debug:
    print("####################  properties in xml file#####################################")
    for prop in typesystem_xml.get_types():
      print(prop)

### Funtion to get unique span list labels names and their counts

In [26]:
# Unique items in name only excluding NONE.
def names_in_span(doc, debug=False):
  # Initialize counts
  span_list_and_counts = {
      'FIGUR': [], 'RAUM': [], 'DING': [], 'SONSTIGE': [],
      'FIGURENGRUPPE': [], 'BEREICH': [], 'GRENZE': [],
      'ABSTRAKTUM': [], 'ZEITABSCHNITT': [], 'ORGANISATION': [], 'NONE': [], 'entities_with_Referenz_Wikidata' : []
  }

  # Process tokens
  for token in doc.select('custom.Span'):  # webanno.custom.Span
      if token.label:  # To prevent error
        if safe_str(token.label)== 'NONE':
          span_list_and_counts['NONE'].append('NONE')  # Add None name to none label
        else:
          if token.label not in span_list_and_counts:
                print('Add new category to Span list!')
          else:
              name_str = safe_str(token.Name).lower()

              if name_str == 'none':
                  span_list_and_counts[token.label].append('NONE')  # Append None label (not unique)
              else:
                  if name_str not in span_list_and_counts[token.label]:  # Check only for unique names
                      span_list_and_counts[token.label].append(name_str)
                      if token.Referenz_Wikidata:
                        if token.Referenz_Wikidata not in span_list_and_counts['entities_with_Referenz_Wikidata']:
                          span_list_and_counts['entities_with_Referenz_Wikidata'].append(token.Referenz_Wikidata)

  if debug:
      print("Unique names in Span:", span_list_and_counts)

  # Create annotation overview table
  for label, names in span_list_and_counts.items():
      annotation_overview_table["Span_" + label] = len(names)  # Count unique names
      if len(names) > 0:
        annotation_overview_table["Span_LISTE_" + label] = ";".join(names) + ";"  # Concatenate names

  if debug:
    print("Annotation overview table:", annotation_overview_table)

  span_list_and_counts_exluding_entities_with_Referenz_Wikidata= result = {k: v for k, v in span_list_and_counts.items() if k != 'entities_with_Referenz_Wikidata'}
  return annotation_overview_table, span_list_and_counts_exluding_entities_with_Referenz_Wikidata

### Count of naming mistakes in span lists.

In [27]:
# Function to count similar words within a list of phrases
def count_similar_words(phrases, threshold, debug):
    similar_count = 0
    list_mistakes = []

    for i in range(len(phrases)):
        for j in range(i + 1, len(phrases)):
            if phrases[j]!='NONE':
              if fuzz.ratio(phrases[i], phrases[j]) > threshold:
                  similar_count += 1
                  list_mistakes.append(phrases[i] + ' vs ' + phrases[j] + ';')
                  if debug:
                      print(f"Similar: '{phrases[i]}' and '{phrases[j]}' with ratio {fuzz.ratio(phrases[i], phrases[j])}")
    return similar_count,list_mistakes




# Function to calculate naming mistakes
def naming_mistakes(input_dict, threshold, debug):
    total_list_naming_mistakes = 0
    all_list_mistakes =  []

    # Process each category separately
    for category, phrases in input_dict.items():
        similar_words_count, list_mistakes = count_similar_words(phrases,threshold,debug)
        if(similar_words_count >0):
          total_list_naming_mistakes += similar_words_count
          all_list_mistakes.extend(list_mistakes)
        if debug:
            print(f"Processing category '{category}':")
            print(f"Number of similar words in '{category}': {similar_words_count}\n")
    stri=''.join(all_list_mistakes)
    return total_list_naming_mistakes, stri

### Additional 5 Categories.

In [28]:
def five_new_categories_info(doc_CAS_xmi,debug):
  #Get the label and count if in the list of Opposition type.
  Erzählerwechsel = 0 #(webanno.custom.Erzhlerwechsel)
  Kritikhaltige_Reflexion = 0 #(webanno.custom.Sonstiges)
  TriggerCodierung = 0 #(webanno.custom.TriggerCodierung)
  TriggerWertung = 0 #(webanno.custom.Trigger)
  TriggerMerkmalsopposition = 0 #(webanno.custom.TriggerMerkmalsopposition)

  for token in doc_CAS_xmi.select('webanno.custom.Erzhlerwechsel'):
    Erzählerwechsel += 1

  for token in doc_CAS_xmi.select('webanno.custom.Sonstiges'):
    Kritikhaltige_Reflexion += 1

  for token in doc_CAS_xmi.select('webanno.custom.TriggerCodierung'):
    TriggerCodierung += 1

  for token in doc_CAS_xmi.select('webanno.custom.Trigger'):
    TriggerWertung += 1

  for token in doc_CAS_xmi.select('webanno.custom.TriggerMerkmalsopposition'):
    TriggerMerkmalsopposition += 1

  if(debug):
    print('five_new_categories_info:', Erzählerwechsel,Kritikhaltige_Reflexion,TriggerCodierung,TriggerWertung,TriggerMerkmalsopposition)

  return(Erzählerwechsel,Kritikhaltige_Reflexion,TriggerCodierung,TriggerWertung,TriggerMerkmalsopposition)


### overlapping oppositions

In [ ]:
def info_overlapping_oppositions(doc_CAS_xmi, debug=False, output_file = 'overlapping_oppositions_errors.txt'):
  overlapping_Oppositionsx = 0
  SPAN_LIST_potential_errorx = {}
  error_countx = 1

  # Open the file in append mode if output_file is provided
  if output_file:
      file = open(output_file, 'a')





  doc_CAS_xmix = doc_CAS_xmi
  # Initialize the dictionary to store governor_name -> dependent_name -> count
  list_governor_name = {}
  filtered_governor_name = {}

  # Loop over the tokens in the document
  for token in doc_CAS_xmix.select('custom.Relation'):
      dependent_name = token.Dependent.Name if token.Dependent else None
      governor_name = token.Governor.Name if token.Governor else None

      # Print if governor_name is the same as dependent_name to correct it in the document.
      if governor_name == dependent_name:

          #print(f"{error_countx}: overlapping oppositions(potential error): Governor Name: {governor_name}, Dependent Name: {dependent_name}")
          error_message = (f"S.No.{error_countx}: overlapping oppositions(potential error): Governor Name: {governor_name}, Dependent Name: {dependent_name}")
          #SPAN_LIST_potential_errorx = (f"Governor Name: {governor_name}, Dependent Name: {dependent_name}")
          # Write the error message to the file
          if error_countx==1:
            Document_titlex = ("================" + annotation_overview_table['Document_title'] + "================"  + '\n')
            file.write(Document_titlex)
          file.write(error_message + '\n')
          error_countx+=1
          continue #Do not add to the list when governor_name == dependent_name.

      # Add to the dictionary: list_governor_name
      if governor_name not in list_governor_name:
          list_governor_name[governor_name] = {}



      if dependent_name in list_governor_name[governor_name]:
          list_governor_name[governor_name][dependent_name] += 1
      else:
          list_governor_name[governor_name][dependent_name] = 1



      # Filter the dictionary: remove entries with a value of 1 and governor_name == dependent_name

      for governor_name, dependents in list_governor_name.items():
          filtered_dependents = {
              dep_name: count for dep_name, count in dependents.items()
              if count > 1 #or governor_name == dep_name
          }
          if filtered_dependents:
              filtered_governor_name[governor_name] = filtered_dependents

      if debug:
        print(f"Governor Name: {governor_name}, Dependent Name: {dependent_name}")


  # Iterate over the dictionary to sum for values for filtered_governor_name.
  for outer_key, inner_dict in filtered_governor_name.items():
      for inner_key, value in inner_dict.items():
          overlapping_Oppositionsx += value

  #print('Unfiltered dictionary',list_governor_name)
  #print('removed entries with a value of 1 unless governor_name == dependent_name:',filtered_governor_name)

  return overlapping_Oppositionsx, filtered_governor_name #, SPAN_LIST_potential_errorx

In [30]:
LIST_ALL_X = {'TRADITIONELL-MODERN': []}  # Initialize as an empty list only used here (Temp)

def info_list_all_AltNeuCodierung(doc_CAS_xmi, debug=False, output_file = 'all_AltNeuCodierung_list.txt'):
  '''
  Retuns: saves file with all AltNeuCodierungs and their counts.
  '''
  SPAN_LIST_oppositionsx = {}

  doc_CAS_xmix = doc_CAS_xmi




  # Iterate through tokens in the selected 'webanno.custom.AltNeuCodierung'
  for token in doc_CAS_xmix.select('webanno.custom.AltNeuCodierung'):

      # Ensure the 'Art' key exists in the dictionary and initialize as a dictionary
      if token.Art not in SPAN_LIST_oppositionsx:
          SPAN_LIST_oppositionsx[token.Art] = {}

      # Handle different cases for Art and map the corresponding Polaritt values
      if token.Art.upper() == 'TIEF-OBERFLÄCHLICH':
          # Initialize the key if not present, then increment
          if token.Polaritt_tief not in SPAN_LIST_oppositionsx[token.Art]:
              SPAN_LIST_oppositionsx[token.Art][token.Polaritt_tief] = 0
          SPAN_LIST_oppositionsx[token.Art][token.Polaritt_tief] += 1  # Increment the count

      elif token.Art.upper() == 'NATÜRLICH-KULTURELL':
          if token.Polaritt_natrlich not in SPAN_LIST_oppositionsx[token.Art]:
              SPAN_LIST_oppositionsx[token.Art][token.Polaritt_natrlich] = 0
          SPAN_LIST_oppositionsx[token.Art][token.Polaritt_natrlich] += 1

      elif token.Art.upper() == 'TRADITIONELL-MODERN':
          if token.Polaritt not in LIST_ALL_X['TRADITIONELL-MODERN']:
            LIST_ALL_X['TRADITIONELL-MODERN'].append(token.Polaritt)

          if token.Polaritt not in SPAN_LIST_oppositionsx[token.Art]:
              SPAN_LIST_oppositionsx[token.Art][token.Polaritt] = 0
          SPAN_LIST_oppositionsx[token.Art][token.Polaritt] += 1



      elif token.Art.upper() == 'HARMONISCH-DISHARMONISCH':
          if token.Polaritt_harmonisch not in SPAN_LIST_oppositionsx[token.Art]:
              SPAN_LIST_oppositionsx[token.Art][token.Polaritt_harmonisch] = 0
          SPAN_LIST_oppositionsx[token.Art][token.Polaritt_harmonisch] += 1

      elif token.Art.upper() == 'GESUND-KRANK':
          if token.Polaritt_gesund not in SPAN_LIST_oppositionsx[token.Art]:
              SPAN_LIST_oppositionsx[token.Art][token.Polaritt_gesund] = 0
          SPAN_LIST_oppositionsx[token.Art][token.Polaritt_gesund] += 1

      else:
          SPAN_LIST_oppositionsx["WARNING:NEW ART"]["WARNING:NEW Polaritt"] = 1
          print("WARNING!!, NEW token.Polaritt category!!")




  if debug:
  # Print the resulting dictionary
    print(SPAN_LIST_oppositionsx)

  # Open the file in append mode if output_file is provided
  if output_file:
      with open(output_file, 'a') as file:
          # Iterate through the dictionary and write to the file in the desired format
          file.write(f"=============={annotation_overview_table['Document_title']}==================\n")
          for art, polaritt_dict in SPAN_LIST_oppositionsx.items():
              for polaritt, count in polaritt_dict.items():
                  file.write(f"'{art}': '{polaritt}': {count},\n")



### evaluative_difference_of_contrastive_entities

In [ ]:
#evaluative_difference_of_contrastive_entities
#IMPORTANT IF NEED TO MAKE CHANGES THEN DUPLICATE IT FOR USING IT FOR TASK 1.1 AS THIS IS THE ONLY DEF ONCE IN THE CODE.
#EXACTLY SAME AS TASK1.1: For each document returns summary_scores in format {('entität', 'FIGUR'): score}

def summary_evaluation(doc_CAS_xmi):

  doc=doc_CAS_xmi
  evaluations = {}
  j = 0
  entity_name = ''
  pos_evaluation_flag = 0
  neg_evaluation_flag = 0
  PIronie_flag = 0 #positive_Wertung_mit_Ironie
  NIronie_flag = 0 #negative_Wertung_mit_Ironie

  # Initialize the dictionary
  dict_positive_entities = {}
  dict_negative_entities = {}
  dict_positive_entities_nameless  = {}
  dict_negative_entities_nameless  = {}

  for segment in doc.select('webanno.custom.Wertung'): ##webanno.custom.rating
      #print("hier: ", segment.TEST)
      #print(str(segment.TEST.elements)) # seems like i don't miss any information by not printing this?
      #print(segment.Label, ": ", segment.get_covered_text(), "; Hinsicht: ", segment.Wertungshinsicht, "; Polarität: ", segment.Polaritt)

      Erzählerwertung = True  ##Narrator Rating = TRUE
      try:
          for e in segment.TEST.elements: # and why does this seem to work here but not above?
              #print(e)
              pass
      except:
          #print("Segment ohne gew. Objekt: Segment without selected object") ##Segment without selected object
          continue

      for e in segment.TEST.elements:
          #print(e.role, ": ", e.target.get_covered_text(), ", Typ: ", e.target.label, ", Name: ", e.target.Name, ", Polarität: ", segment.Polaritt)

          pos_evaluation_flag = 0
          neg_evaluation_flag = 0
          PIronie_flag = 0
          NIronie_flag = 0
          entity_name = ''

          if(segment.Wertungshinsicht == "eudämonistisch"): #Exluded all eudämonistisch evaluation for FIGUR and FIGURENGRUPPE
            if (e.target.label == "FIGURENGRUPPE") or (e.target.label == "FIGUR"):
              continue
          # create lists of positive entities
          if (Erzählerwertung == True) and (segment.Polaritt == "sehr positiv" or segment.Polaritt == "positiv") and (e.role == "gewertete Entität"):
              if e.target.Name: #Not included: e.target.Name == None
                entity_name = e.target.Name.lower()
                ent_typ = e.target.label
                pos_evaluation_flag = 1
                if(segment.Ironie):
                  PIronie_flag = 1

          # create lists of negative entities
          elif (Erzählerwertung == True) and (segment.Polaritt == "sehr negativ" or segment.Polaritt == "negativ") and (e.role == "gewertete Entität"):
              if e.target.Name: #Not included: e.target.Name == None
                entity_name = e.target.Name.lower()
                ent_typ = e.target.label
                neg_evaluation_flag = 1
                if(segment.Ironie):
                  NIronie_flag = 1

          if entity_name:
            j = j + 1
            evaluations[j] = entity_name, ent_typ, pos_evaluation_flag, neg_evaluation_flag, PIronie_flag, NIronie_flag

    # Test
  #print('========================')
  #for k,v in evaluations.items():
    #print(k, ": ", v)
  #print('========================')


  # Initialize a defaultdict where each entity maps to another defaultdict (for ent_typ)
  grouped_evaluations = defaultdict(lambda: defaultdict(lambda: {'positive': 0, 'negative': 0, 'PIronie': 0, 'NIronie': 0}))
  total_evaluations = 0

  # Iterate over encodings
  for _, (entity_name, ent_typ, pos_evaluation_flag, neg_evaluation_flag, PIronie_flag, NIronie_flag) in evaluations.items():
      grouped_evaluations[entity_name][ent_typ]['positive'] += pos_evaluation_flag #- PIronie_flag + NIronie_flag
      grouped_evaluations[entity_name][ent_typ]['negative'] += neg_evaluation_flag #+ PIronie_flag - NIronie_flag
      grouped_evaluations[entity_name][ent_typ]['PIronie'] += PIronie_flag
      grouped_evaluations[entity_name][ent_typ]['NIronie'] += NIronie_flag
      total_evaluations += pos_evaluation_flag + neg_evaluation_flag

  # Test
  #print('========================')
  #for k,v in grouped_evaluations.items():
    #print(k, ": ", v, ":",total_evaluations)
  #print('========================')

    # Calculate the sum of all positive and negative evaluations across all entities and ent_typ (same as total_evaluations, just to check)
  total_entity_evaluations = sum(
        evaluations_counts['positive'] + evaluations_counts['negative']
        for entity_evaluations in grouped_evaluations.values()
        for evaluations_counts in entity_evaluations.values()
    )
  #print("total_entity_evaluations",total_entity_evaluations)

  # Calculate summary encoding score for every entity and ent_typ
  summary_scores = {}

  if total_entity_evaluations > 0:
    for entity_name, entity_evaluations in grouped_evaluations.items():
        for ent_typ, evaluations_counts in entity_evaluations.items():
          score = (evaluations_counts['positive'] + evaluations_counts['NIronie'] - evaluations_counts['negative'] - evaluations_counts['PIronie']) / total_entity_evaluations
          summary_scores[(entity_name, ent_typ)] = score

  # Output summary scores for each entity and ent_typ
  #print("##########################")
  #print("summary_scores",summary_scores)
  #print("##########################")
  #print("##########################")
  #print("grouped_evaluations",grouped_evaluations)
  #print("##########################")
  #print("Task1.1: ===",annotation_overview_table['Document_title'], summary_scores)
  #print("Task1.1: ===",annotation_overview_table['Document_title'], grouped_evaluations)
  return (summary_scores,grouped_evaluations)

In [32]:
# Function to calculate evaluation for a given entity
def calculate_evaluation(evaluations_counts, total_entity_evaluations_in_doc):
    total_evaluations = total_entity_evaluations_in_doc

    if total_evaluations == 0:
        return 0
    result = (
        (evaluations_counts['positive'] + evaluations_counts['NIronie'] -
         evaluations_counts['negative'] - evaluations_counts['PIronie']) /
        total_evaluations
    )
    print("From:def:calculate_evaluation", evaluations_counts, total_entity_evaluations_in_doc,result)
    return result


# Generalized function to calculate the difference and mean based on Span_LISTE_Overlapping_Oppositions
def compute_evaluation_differences(Span_LISTE_Overlapping_Oppositions, grouped_evaluationsx):

    # Convert dictionary items of Span_LISTE_Overlapping_Oppositions to lowercase
    Span_LISTE_Overlapping_Oppositions = {k.lower(): {kk.lower(): vv for kk, vv in v.items()} for k, v in annotation_overview_table['Span_LISTE_Overlapping_Oppositions'].items()}

    # Calculate the sum of all positive and negative evaluations across all entities and ent_typ (same as total_evaluations, just to check)
    total_entity_evaluations_in_doc = sum(
        evaluations_counts['positive'] + evaluations_counts['negative']
        for entity_evaluations in grouped_evaluationsx.values()
        for evaluations_counts in entity_evaluations.values()
    )
    differences = []

    for main_entity, opposing_entities in Span_LISTE_Overlapping_Oppositions.items():
        for opposing_entity, _ in opposing_entities.items():
            # Get evaluation counts for both the main entity and its opposing entity
            # Check if the main_entity is present in grouped_evaluationsx and has data
            if main_entity in grouped_evaluationsx and grouped_evaluationsx[main_entity]:
                main_entity_evaluation_counts = grouped_evaluationsx[main_entity].get(next(iter(grouped_evaluationsx[main_entity])), defaultdict(int))
            else:
                main_entity_evaluation_counts = defaultdict(int)  # Use default if not found

            # Do the same for opposing_entity
            if opposing_entity in grouped_evaluationsx and grouped_evaluationsx[opposing_entity]:
                opposing_entity_evaluation_counts = grouped_evaluationsx[opposing_entity].get(next(iter(grouped_evaluationsx[opposing_entity])), defaultdict(int))
            else:
                opposing_entity_evaluation_counts = defaultdict(int)





            # Calculate evaluations for each entity
            main_entity_evaluation = calculate_evaluation(main_entity_evaluation_counts, total_entity_evaluations_in_doc)
            opposing_entity_evaluation = calculate_evaluation(opposing_entity_evaluation_counts, total_entity_evaluations_in_doc)

            # Calculate the difference
            difference = main_entity_evaluation - opposing_entity_evaluation
            differences.append(difference)

    # Calculate and return the mean of the differences
    if differences:
        mean_difference = abs(sum(differences) / len(differences)) #01-11-2024: evaluative_difference_of_contrastive_entities in annotation overview; this should be the absolute value
    else:
        mean_difference = 0
    print("Differences:", differences, "Mean Difference:", mean_difference)
    return differences, mean_difference

### All encodings and wetung with sehr

In [33]:
def find_sehr_in_encodings_wertung(doc, debug=False):
  index = 0
  encodings_with_sher= {}
  for token in doc.select('webanno.custom.AltNeuCodierung'): ##webanno.custom.rating

    # Handle different cases for Art and map the corresponding Polaritt values
    #Modified with try bocks to prevent errors in new files. 06-04-25


    for token in doc.select('webanno.custom.AltNeuCodierung'):  # webanno.custom.rating

        try:
            if token.Art.upper() == 'TIEF-OBERFLÄCHLICH':
            # Initialize the key if not present, then increment

                if "sehr" in token.Polaritt_tief:
                    encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt_tief, token.get_covered_text)
                    index += 1

            elif token.Art.upper() == 'NATÜRLICH-KULTURELL':
                if "sehr" in token.Polaritt_natrlich:
                    encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt_natrlich, token.get_covered_text)
                    index += 1

            elif token.Art.upper() == 'TRADITIONELL-MODERN':
                if "sehr" in token.Polaritt:
                    encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt, token.get_covered_text)
                    index += 1

            elif token.Art.upper() == 'HARMONISCH-DISHARMONISCH':
                if "sehr" in token.Polaritt_harmonisch:
                    encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt_harmonisch, token.get_covered_text)
                    index += 1

            elif token.Art.upper() == 'GESUND-KRANK':
                if "sehr" in token.Polaritt_gesund:
                    encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt_gesund, token.get_covered_text)
                    index += 1

            else:
                print("WARNING!! NEW token.Polaritt category!!")
                try:
                    if "sehr" in token.Polaritt:
                        encodings_with_sher[index] = ("segment.Polaritt:", token.Polaritt, token.get_covered_text)
                        index += 1
                except Exception as e:
                    # print("Segment ohne gew. Objekt: Segment without selected object") ##Segment without selected object
                    continue

        except Exception as e:
            print(f"Error processing token {token}: {e}")
            continue






  current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
  output_file="sehr_in_AltNeuCodierung"+current_time+".txt"
  # Open the file in append mode if output_file is provided
  if output_file:
      with open(output_file, 'a') as file:
          if encodings_with_sher:
            file.write(f"=============={annotation_overview_table['Document_title']}==================\n")
          # Iterate through the dictionary and write to the file in the desired format
          for i in  encodings_with_sher.items():
            file.write(f"{i}:\n")



  relations_with_sher= {}
  index=0
  for segment in doc.select('webanno.custom.Wertung'): ##webanno.custom.rating
      #print("hier: ", segment.TEST)
      #print(str(segment.TEST.elements)) # seems like i don't miss any information by not printing this?
      #print(segment.Label, ": ", segment.get_covered_text(), "; Hinsicht: ", segment.Wertungshinsicht, "; Polarität: ", segment.Polaritt)

      try:
          for e in segment.TEST.elements: # and why does this seem to work here but not above?
              #print(e)
              pass
      except:
          #print("Segment ohne gew. Objekt: Segment without selected object") ##Segment without selected object
          continue

      for e in segment.TEST.elements:
          #print(e.role, ": ", e.target.get_covered_text(), ", Typ: ", e.target.label, ", Name: ", e.target.Name, ", Polarität: ", segment.Polaritt)

          # create lists of positive entities
          if "sehr" in segment.Polaritt:
              relations_with_sher[index]  = ("segment.Polaritt:",segment.Polaritt, "segment.TEST.elements:",e)
              index+=1

  current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
  output_file="sehr_in_Wertung"+current_time+".txt"

  # Open the file in append mode if output_file is provided
  if output_file:
      with open(output_file, 'a') as file:
          if relations_with_sher:
            file.write(f"=============={annotation_overview_table['Document_title']}==================\n")

          # Iterate through the dictionary and write to the file in the desired format
          for i in  relations_with_sher.items():
            file.write(f"{i}:\n")

### duplicate span for entities.

In [34]:
def find_dupicate_span_in_entities(doc, debug=False):

  # Initialize dictionaries
  span_name_begin_end = {}
  same_span_list = {}

  # Helper function to safely convert values to strings
  def safe_str(value):
      return str(value) if value is not None else ''

  # Processing tokens to populate span_name_begin_end
  for token in doc.select('custom.Span'):  # webanno.custom.Span
      if token.label:  # Check if label exists
          # Create a unique key using Name, begin, and end attributes
          key = "name:"+safe_str(token.Name).lower() +" begin:"+ str(token.begin) +" end:"+ str(token.end)

          # Check if the token is already in span_name_begin_end
          if key not in span_name_begin_end:
              span_name_begin_end[key] = token
          else:
              # If it's already in span_name_begin_end, add it to same_span_list if not already present
              if key not in same_span_list:
                  same_span_list[key] = token


  current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
  output_file="duplicate_span_for_entities"+"_"+current_time+".txt"
    # Open the file in append mode if output_file is provided
  if output_file:
      with open(output_file, 'a') as file:
          # Iterate through the dictionary and write to the file in the desired format
          file.write(f"=============={annotation_overview_table['Document_title']}==================\n")

          for i in  same_span_list.items():
            file.write(f"{i}:\n")

  # Print the results
  if(debug):
    print("Span Name Begin End Dictionary:", span_name_begin_end)
    print("Same Span List Dictionary (Duplicates):", same_span_list)


### counts of Codierung & Wertung both with and without Trigger.(wrt. span)

In [35]:
def count_Codierung_Wertung_with_and_without_trigger(doc, debug=False):
  '''
  Retuns: codierun_with_trigger, codierun_without_trigger,  wertung_with_trigger, wertung_without_trigger
  '''
  codierun_with_trigger=0
  codierun_without_trigger=0
  trigger_found=False

  for token in doc.select('webanno.custom.AltNeuCodierung'):
    for _trigger in doc.select('webanno.custom.TriggerCodierung'):
      if(_trigger.begin >= token.begin) and (_trigger.end <= token.end):
        codierun_with_trigger+=1
        trigger_found=True
        break
    if not trigger_found:
      codierun_without_trigger+=1
    trigger_found=False



  if debug:
    print("=====count_Codierung_Wertung_with_and_without_trigger=====")
    print('codierun_with_trigger',codierun_with_trigger)
    print('codierun_without_trigger',codierun_without_trigger)



  wertung_with_trigger = 0
  wertung_without_trigger = 0
  trigger_found=False

  for token in doc.select('webanno.custom.Wertung'): ##webanno.custom.rating
    for w_trigger in doc.select('webanno.custom.Trigger'): # Evaluation/Wertung Trigger
      if(w_trigger.begin)>=token.begin and (w_trigger.end)<=token.end:
        wertung_with_trigger+=1
        trigger_found=True
        break
    if not trigger_found:
      wertung_without_trigger+=1
    trigger_found=False

  if debug:
    print("=====count_Codierung_Wertung_with_and_without_trigger=====")
    print("wertung_with_trigger: ",wertung_with_trigger)
    print("wertung_without_trigger: ",wertung_without_trigger)


  return codierun_with_trigger, codierun_without_trigger, wertung_with_trigger, wertung_without_trigger

### opposition tokens.(wrt. span) used in  descriptive statistics

In [36]:
#Added: 08-12-2024
def info_oppositions_tokens_desctiptive_stats(doc_CAS_xmi, debug=False):
  if(debug):
    print("=====info_oppositions_tokens_desctiptive_stats=====")

  doc_CAS_xmix = doc_CAS_xmi
  # Initialize the dictionary to store governor_name -> dependent_name -> count
  list_oppositions_governor_dependent_namex = {}
  count_Oppositionsx =0
  count_dependent_tokens = 0

  # Loop over the tokens in the document
  for token in doc_CAS_xmix.select('custom.Relation'):
      governor_name = token.Governor.Name if token.Governor else None
      dependent_name = token.Dependent.Name if token.Dependent else None

      # Add to the dictionary: list_governor_name
      if governor_name not in list_oppositions_governor_dependent_namex:
          list_oppositions_governor_dependent_namex[governor_name] = {}

      if dependent_name in list_oppositions_governor_dependent_namex[governor_name]:
          list_oppositions_governor_dependent_namex[governor_name][dependent_name] += 1
      else:
          list_oppositions_governor_dependent_namex[governor_name][dependent_name] = 1

      if debug:
        print(f"Governor Name: {governor_name}, Dependent Name: {dependent_name}")


      #Count no of tokens with in the oppositions
      for doc_tokens in doc_CAS_xmix.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'):
        if(doc_tokens.begin)>=token.begin and (doc_tokens.end)<=token.end:
          count_dependent_tokens+=1

  # Iterate over the dictionary to sum for values for filtered_governor_name.
  for outer_key, inner_dict in list_oppositions_governor_dependent_namex.items():
      for inner_key, value in inner_dict.items():
          count_Oppositionsx += value

  if debug:
    print('list_oppositions_governor_dependent_name',list_oppositions_governor_dependent_namex)
    print('count_Oppositionsx',count_Oppositionsx)
    print('count_dependent_tokens:it is actually = Oppositionsx tokens',count_dependent_tokens) # it is actually = Oppositionsx tokens

  return list_oppositions_governor_dependent_namex,count_Oppositionsx, count_dependent_tokens

In [37]:
#19-12-2024
def info_Wertung_n_and_tokens_desctiptive_stats(doc_CAS_xmi, debug=False):

  if(debug):
    print("=====info_Wertung_n_and_tokens_desctiptive_stats=====")

  doc_CAS_xmix = doc_CAS_xmi
  n=0
  count_dependent_tokens = 0
  for token in doc_CAS_xmix.select('webanno.custom.Wertung'):
    n+=1
    #Count no of tokens with in the encodings
    for doc_tokens in doc_CAS_xmix.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'):
      if(doc_tokens.begin)>=token.begin and (doc_tokens.end)<=token.end:
        count_dependent_tokens+=1

  return n,count_dependent_tokens

In [38]:
#19-12-2024
def info_encodings_n_and_tokens_desctiptive_stats(doc_CAS_xmi, debug=False):
  if(debug):
    print("=====info_encodings_n_and_tokens_desctiptive_stats=====")

  doc_CAS_xmix = doc_CAS_xmi
  n=0
  count_dependent_tokens = 0
  for token in doc_CAS_xmix.select('webanno.custom.AltNeuCodierung'):
    n+=1
    #Count no of tokens with in the encodings
    for doc_tokens in doc_CAS_xmix.select('de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'):
      if(doc_tokens.begin)>=token.begin and (doc_tokens.end)<=token.end:
        count_dependent_tokens+=1

  return n,count_dependent_tokens

### list of all documents/spans that have Wertungshinsicht "nicht spezifiziert"

In [39]:
def docs_with_nicht_spezifiziert(doc):
  #Get the label and count if in the list of Evaluation types.
  file_path="docs_with_nicht_spezifiziert.txt"
  new_doc=True
  for token in doc.select('webanno.custom.Wertung'):
    if token.Wertungshinsicht =='nicht spezifiziert':
      if(new_doc):
        textx = "================"+str(annotation_overview_table['Document_title'])+"================"
        new_doc = False
        try:
          with open(file_path, 'a') as file:  # 'a' mode opens the file for appending
              file.write(textx + '\n')  # Appending text with a newline for clarity
        except Exception as e:
            print(f"An error occurred: {e}")

      textx=token.get_covered_text() +"::::::" +str(token)
      try:
          with open(file_path, 'a') as file:  # 'a' mode opens the file for appending
              file.write(textx + '\n')  # Appending text with a newline for clarity
      except Exception as e:
          print(f"An error occurred: {e}")

###  eudamonistic evaluations that have more than one evaluated entity

In [41]:
def eudämonistisch_list_counts(doc_CAS_xmi):
    _eudämonistisch_counts = []
    _eudämonistisch_counts_coveredtext = []
    _gewertete_entitat_count = 0
    #At least one evaluated entity should be of type  "FIGUR" AND not all evaluated entities should be "FIGUR" 11-04-2025
    _figure_count = 0
    _entities_count = 0

    for wertung_token in doc_CAS_xmi.select('webanno.custom.Wertung'):
        if wertung_token.Wertungshinsicht == 'eudämonistisch':
            _gewertete_entitat_count = 0

            _first_wertung_token = ''
            _added_wertung_token = False
            _added_first_wertung_token = False

            _figure_count = 0
            _entities_count = 0

            try:
                for item in wertung_token.TEST.elements:
                  if item.role == 'gewertete Entität':
                    _entities_count += 1
                    if item.label == 'FIGUR':
                      _figure_count += 1

                if _entities_count > 1 and _figure_count < _entities_count:
                  pass
                else:
                  continue

                _figure_count = 0
                _entities_count = 0


                for item in wertung_token.TEST.elements:
                    if item.role == 'gewertete Entität':
                        _gewertete_entitat_count += 1

                        if not _added_first_wertung_token:
                          _added_first_wertung_token = True
                          _first_wertung_token = item.get_covered_text()


                    if _gewertete_entitat_count > 1:
                      if item.role == 'gewertete Entität':


                        if(_added_first_wertung_token & len(_first_wertung_token)>0):
                            _eudämonistisch_counts_coveredtext.append(_first_wertung_token)
                            _first_wertung_token = ''
                        _eudämonistisch_counts_coveredtext.append(item.get_covered_text())
                      if not _added_wertung_token:
                          _eudämonistisch_counts.append(wertung_token)
                          _added_wertung_token = True
            except Exception as e:
                print(f"def_eudämonistisch_list_counts: Error processing wertung_token.TEST.elements: {e}")

    return _eudämonistisch_counts,_eudämonistisch_counts_coveredtext


In [42]:
def eudämonistisch_list_counts(doc_CAS_xmi):
    _eudämonistisch_counts = []
    _eudämonistisch_counts_coveredtext = []
    _eudämonistisch_elements_coveredtext = []
    _gewertete_entitat_count = 0
    #At least one evaluated entity should be of type  "FIGUR" AND not all evaluated entities should be "FIGUR" 11-04-2025
    _figure_count = 0
    _entities_count = 0

    for wertung_token in doc_CAS_xmi.select('webanno.custom.Wertung'):
        if wertung_token.Wertungshinsicht == 'eudämonistisch':
            _gewertete_entitat_count = 0
            _added_wertung_token = False

            _added_first_wertung_token = False
            _first_wertung_token = ''
            _first_item = ''
            _figure_count = 0
            _entities_count = 0
            try:
                for item in wertung_token.TEST.elements:
                  print("item::::",item)
                  if item.role == 'gewertete Entität':
                    _entities_count += 1
                    if item.target.label == 'FIGUR':
                      _figure_count += 1

                if _entities_count > 1 and _figure_count < _entities_count and _figure_count>=1:
                  pass
                else:
                  continue

                _figure_count = 0
                _entities_count = 0



                for item in wertung_token.TEST.elements:
                    if item.role == 'gewertete Entität':
                        _gewertete_entitat_count += 1
                    if _gewertete_entitat_count > 1:
                        if not _added_wertung_token:
                            _eudämonistisch_counts.append(wertung_token)
                            _added_wertung_token = True

                    if item.role == 'gewertete Entität':
                        if not _added_first_wertung_token:
                          _added_first_wertung_token = True
                          _first_wertung_token = wertung_token.get_covered_text()
                          _first_item =item.target.get_covered_text()

                    if _gewertete_entitat_count > 1:
                      if item.role == 'gewertete Entität':
                        if(_added_first_wertung_token and len(_first_wertung_token)>0):
                            _eudämonistisch_counts_coveredtext.append(_first_wertung_token)
                            _first_wertung_token = ''
                            _eudämonistisch_elements_coveredtext.append(_first_item)
                            _first_item = ''
                        _eudämonistisch_elements_coveredtext.append(item.target.get_covered_text())
                        _eudämonistisch_counts_coveredtext.append(wertung_token.get_covered_text())

            except Exception as e:
                print(f"def_eudämonistisch_list_counts: Error processing wertung_token.TEST.elements: {e}")
    #returns wertung_token, wertung_token covered text, wertung test elements covered text.
    return _eudämonistisch_counts,_eudämonistisch_counts_coveredtext,_eudämonistisch_elements_coveredtext

### All above functions are called here for simplicity

In [ ]:
#Here all methods are implemented.

def perform_analysis(typesystem_path,cas_file_path,debug, threshold =90):
  with open(typesystem_path, 'rb') as f:
    typesystem_xml = load_typesystem(f)

  with open(cas_file_path, 'rb') as f:
    doc_CAS_xmi = load_cas_from_xmi(f, typesystem=typesystem_xml) # cassis requires xml 1.0  not 1.1! somehow it seems to work nevertheless?


  print("======================= START =================================")

  #Reset to defauls
  annotation_overview_table = reset_annotation_overview_table(debug)

  #Get Document info to recheck the title and other properties.
  _doc_title= document_metadata_info(typesystem_xml, doc_CAS_xmi, debug)

  #To view properties in xml. (Not essential part of code.)
  view_xml_properties(typesystem_xml,debug)

  #Token count: Number of tokens
  #'name=de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Token'
  #    <type5:Token xmi:id="15619" sofa="1" begin="6689" end="6691" order="0"/>
  annotation_overview_table['Textlänge_token'] = token_info(doc_CAS_xmi,debug) #Text length (tokens)

  #Sentence count
  #'de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Sentence'
  #    <type5:Sentence xmi:id="84858" sofa="1" begin="2099" end="2238"/>
  sen_info(doc_CAS_xmi,debug)

  #Entity info
  #'name=custom.Span'
  #    <custom:Span xmi:id="86999" sofa="1" begin="25229" end="25238" label="DING"/>
  entities_info(doc_CAS_xmi,_doc_title,debug)

  #Ratings/Evaluation (Positive and negative) count
  #'webanno.custom.Wertung'
  #<custom2:Wertung xmi:id="12714" sofa="1" begin="3042" end="3185" TEST="12733" Wertungshinsicht="moralisch" Polaritt="negativ" Ironie="false"/>
  #<custom2:Wertung xmi:id="30460" sofa="1" begin="2824" end="2896" TEST="30492 30495" Wertungshinsicht="sozialer Status" Polaritt="negativ" Unsicherheitsmarker="unsicher gewertete Entität" Ironie="false"/>
  #+
  #<custom2:WertungTESTLink xmi:id="12269" role="gewertete Entität" target="12261"/>
  #w.c.WertungTESTLink(role=gewertete Entität, target=c.Span(Referenz_Wikidata=http://www.wikidata.org/entity/Q298, begin=113, end=171, label=ZEITABSCHNITT))
  evaluations_count_positive_negative(doc_CAS_xmi,debug)


  #Count Analysis of Ratings/Evaluation (Exluding positive and negative counts)
  #'webanno.custom.Wertung'
  #<custom2:Wertung xmi:id="43072" sofa="1" begin="311" end="332" TEST="43660" Wertungshinsicht="epistemisch" Polaritt="sehr positiv" Ironie="false"/>
  evaluation_info(doc_CAS_xmi,debug)

  #Coding Evaluation
  #'webanno.custom.AltNeuCodierung'
  #<custom2:AltNeuCodierung xmi:id="30804" sofa="1" begin="4545" end="4689" Bezugsentitt="30825" Art="Tief-Oberflächlich" Polaritt_tief="tief/tiefsinnig"/>
  #<custom2:AltNeuCodierung xmi:id="31113" sofa="1" begin="6899" end="7033" Art="Tief-Oberflächlich" Polaritt_tief="sehr tief/tiefsinnig"/>
  coding_info(doc_CAS_xmi,debug)

  #Count Analysis of Opposition
  #'custom.Relation'
  #    <custom:Relation xmi:id="89960" sofa="1" begin="49" end="95" Dependent="87533" Governor="88103" label="Natürlich-kulturell-Opposition"/>
  #    <custom:Relation xmi:id="10571" sofa="1" begin="669" end="702" Dependent="9735" Governor="9416" label="Sonstige Merkmalsopposition" Merkmalsopposition="Mann vs. Frau"/>
  opposition_info(doc_CAS_xmi,debug)


  #Count Analysis of group of characters/family
  character_family_info(doc_CAS_xmi,debug)

  #List of unique "Names" from "custom:Span" and labels and their counts separated by ";".
  names_in_span(doc_CAS_xmi, debug)

  #<type:LayerDefinition xmi:id="13033" name="webanno.custom.Sonstiges" uiName="Kritikhaltige Reflexion"/>
  Kritikhaltige_Reflexion_defined(doc_CAS_xmi)



  #Count of total naming mistakes
  annotation_overview_table, input_dict = names_in_span(doc_CAS_xmi, debug)

  annotation_overview_table['total_list_naming_mistakes'],annotation_overview_table['list_naming_mistakes']=naming_mistakes(input_dict,threshold , debug)

  annotation_overview_table['Wertung_eudämonistisch_figure'],annotation_overview_table['Wertung_eudämonistisch_not_figure'] = eudämonistisch_fig_notfig_counts(doc_CAS_xmi)


  #Five new categories
  #1. "Erzählerwechsel" (webanno.custom.Erzhlerwechsel) #2. "Kritikhaltige Reflexion" (webanno.custom.Sonstiges);
  #3. TriggerCodierung (webanno.custom.TriggerCodierung); #4. TriggerWertung (webanno.custom.Trigger);   #5. TriggerMerkmalsopposition (webanno.custom.TriggerMerkmalsopposition)
  annotation_overview_table['Erzählerwechsel'], annotation_overview_table['Kritikhaltige_Reflexion'], annotation_overview_table['TriggerCodierung'], annotation_overview_table['TriggerWertung'], annotation_overview_table['TriggerMerkmalsopposition'] = five_new_categories_info(doc_CAS_xmi,debug)


  #Overlapping Oppostions
  current_timex = datetime.now(tz_Berlin).strftime('%Y%m%d')
  output_filex="overlapping_oppositions_errors_"+current_timex+".txt"
  annotation_overview_table['Overlapping_Oppositions'], annotation_overview_table['Span_LISTE_Overlapping_Oppositions'] = info_overlapping_oppositions(doc_CAS_xmi, debug,output_filex)


  #for AltNeuCodierung list added 04-10-2024 to check for errors.
  current_timex = datetime.now(tz_Berlin).strftime('%Y%m%d')
  output_filex="all_AltNeuCodierung_list_"+current_timex+".txt"
  info_list_all_AltNeuCodierung(doc_CAS_xmi, debug, output_filex)



  #evaluative_difference_of_contrastive_entities
  _, grouped_evaluationsx = summary_evaluation(doc_CAS_xmi)
  differences, annotation_overview_table['evaluative_difference_of_contrastive_entities'] = compute_evaluation_differences(annotation_overview_table['Span_LISTE_Overlapping_Oppositions'], grouped_evaluationsx)

  docs_with_nicht_spezifiziert(doc_CAS_xmi)

  find_sehr_in_encodings_wertung(doc_CAS_xmi, debug=False) 
  find_dupicate_span_in_entities(doc_CAS_xmi, debug=False)
  annotation_overview_table['codierung_with_trigger_explicit'], annotation_overview_table['codierung_without_trigger_implicit'],annotation_overview_table['wertung_with_trigger_explicit'], annotation_overview_table['wertung_without_trigger_implicit'] = count_Codierung_Wertung_with_and_without_trigger(doc_CAS_xmi, debug=False)#Added:31-10-24
  if(annotation_overview_table['codierung_without_trigger_implicit']==0):
    annotation_overview_table['ratio_explicit_implicit_codierung']=0
  else:
    annotation_overview_table['ratio_explicit_implicit_codierung'] = annotation_overview_table['codierung_with_trigger_explicit']/annotation_overview_table['codierung_without_trigger_implicit']

  if(annotation_overview_table['wertung_without_trigger_implicit']==0):
    annotation_overview_table['ratio_explicit_implicit_wertung']=0
  else:
    annotation_overview_table['ratio_explicit_implicit_wertung'] = annotation_overview_table['wertung_with_trigger_explicit']/annotation_overview_table['wertung_without_trigger_implicit']
  print('\n')


  list_oppositions_governor_dependent_namex,count_Oppositionsx,oppositions_tokens = info_oppositions_tokens_desctiptive_stats(doc_CAS_xmi,debug=False) #Added for descriptive statistics: 08-12-2024
  stats_info = {'Doc_title':'','list_oppositions_governor_dependent_name':'','count_Oppositionsx':0, 'oppositions_tokens':0}
  stats_info['Doc_title'] = annotation_overview_table['Document_title']
  stats_info['list_oppositions_governor_dependent_name'] = list_oppositions_governor_dependent_namex
  stats_info['count_Oppositionsx'] = count_Oppositionsx
  stats_info['oppositions_tokens'] = oppositions_tokens

  stats_info['n_encodings'] , stats_info['tokens_encodings']= info_encodings_n_and_tokens_desctiptive_stats(doc_CAS_xmi,debug=False) #Added for descriptive statistics: 19-12-2024
  stats_info['n_evaluations'] , stats_info['tokens_evaluations']= info_Wertung_n_and_tokens_desctiptive_stats(doc_CAS_xmi,debug=False) #Added for descriptive statistics: 19-12-2024

  _list_eudämonistisch_counts = []
  _eudämonistisch_counts_coveredtext =[]
  _eudämonistisch_elements_coveredtext = []
  _list_eudämonistisch_counts,_eudämonistisch_counts_coveredtext,_eudämonistisch_elements_coveredtext = eudämonistisch_list_counts(doc_CAS_xmi)
  #print("_list_eudämonistisch_counts,_eudämonistisch_counts_coveredtext,_eudämonistisch_elements_coveredtext",_list_eudämonistisch_counts,_eudämonistisch_counts_coveredtext,_eudämonistisch_elements_coveredtext)

  if len(_list_eudämonistisch_counts) > 0:
      current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
      output_file="eudamonistic_withmore_evaluations"+"_"+current_time+".txt"
      with open(output_file, "a", encoding="utf-8") as f:
          f.write(f"================ {annotation_overview_table['Document_title']} ================\n")
          for item in _list_eudämonistisch_counts:
              f.write(f"{item}\n\n")  # Double newline for spacing
          f.write("\n")  # Add spacing after the document block



  if len(_eudämonistisch_counts_coveredtext) > 0:
      current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
      output_file="eudamonistisch_Wertung_coveredtext"+"_"+current_time+".txt"
      with open(output_file, "a", encoding="utf-8") as f:
          f.write(f"================ {annotation_overview_table['Document_title']} ================\n")
          for item in _eudämonistisch_counts_coveredtext:
              f.write(f"{item}\n\n")  # Double newline for spacing
          f.write("\n")  # Add spacing after the document block



  if len(_eudämonistisch_elements_coveredtext) > 0:
      current_time = datetime.now(tz_Berlin).strftime('%Y%m%d')
      output_file="eudamonistisch_target_test_elements_coveredtext"+"_"+current_time+".txt"
      with open(output_file, "a", encoding="utf-8") as f:
          f.write(f"================ {annotation_overview_table['Document_title']} ================\n")
          for item in _eudämonistisch_elements_coveredtext:
              f.write(f"{item}\n\n")  # Double newline for spacing
          f.write("\n")  # Add spacing after the document block

  save_to_csv(stats_info,'00_oppositions_descriptive_stats_info.csv') #Data saved as seperate file and is not saved in main annotation table as it is only required in descriptive statistics part.

  print("#################### Annotation Overview Scheme #####################################")
  print(annotation_overview_table)
  print('\n')

  if(annotation_overview_table['Document_title']!= 'Brecht_Das_Wiedersehen%20-%20Kopie.txt'):
    save_to_csv(annotation_overview_table)

  #print('To check: Erzählerwertung	Figurenwertung	Wertung_Kompositionsebene	positive_Wertung	negative_Wertung')
  print("===================== END ===================================")



## Main cell to call the funtions

In [ ]:
#Main cell for getting and analyzing data from annotated files

# Path to the folder in Google Drive where the files are stored
drive_folder_path = '/content/'

# delete all files
action_delete_all_files(drive_folder_path)


if PROCESS_SINGLE_AUTHOR:
  print('upload inception annotated zip of SINGLE author')
  upload_next_author_zip()

  print('\n')

  # Reading a Common Analysis System (CAS) file
  typesystem_path = drive_folder_path + "TypeSystem.xml"
  cas_file_path = drive_folder_path + "CURATION_USER.xmi"

  perform_analysis(typesystem_path,cas_file_path,DEBUG, THRESHOLD)

else:
  print('upload inception annotated zip containing ALL authors')
  #Upload the ZIP file
  uploaded_zip = files.upload()

  # Extract the filename from the uploaded files
  zip_filename = list(uploaded_zip.keys())[0]

  # Extraction path
  extract_path = './'

  extract_nested_zip(zip_filename,'./')

  # Extract files
  extract_nested_zip(zip_filename, extract_path)

  # Get folder names
  folders = get_folder_names()

  for folder in folders:
    # Reading a Common Analysis System (CAS) file
    typesystem_path = drive_folder_path + 'curation/' + folder + "/TypeSystem.xml"
    cas_file_path = drive_folder_path + 'curation/' + folder + "/CURATION_USER.xmi"
    perform_analysis(typesystem_path,cas_file_path,DEBUG, THRESHOLD)


Deleted directory: /content/.config
Deleted directory: /content/sample_data
upload inception annotated zip containing ALL authors


Saving curated-docs-evaluative-structures-and-cultural-cri-1-2025-05-26-132454.zip to curated-docs-evaluative-structures-and-cultural-cri-1-2025-05-26-132454.zip
======================= START =================================
####################  Wertung (Analysis of Ratings ) : Excluding positive_Wertung	negative_Wertung #####################################


Figurenwertung 43
Erzählerwertung 25


Unfiltered dictionary {'Grimm: eine besondere Wohnung': {'Grimm: die Werkstätte': 1}, 'Grimm: unsere Höhle': {'Grimm: die Werkstätte': 1}, 'Grimm: ein Schneiderlein': {'Grimm: ein gewaltiger Riese': 1, 'Grimm: die Kriegsleute': 1}, 'Grimm: seine einzige Tochter': {'Grimm: dem König': 1}}
removed entries with a value of 1 unless governor_name == dependent_name: {}
Task1.1: === Grimm_Das_tapfere_Schneiderlein.txt {('grimm: gut mus', 'DING'): 0.05084745762711865, ('grimm: eine bauersfrau', 'FIGUR'): 0.03389830508474576, ('grimm: ihrem schweren korbe', 'DING'): -0.01694915254237288, ('grimm: d

## Save annoation overview scheme

In [ ]:
original_csv = '01_output.csv'  # Path to the original CSV file
check_for_file(original_csv)

# Path where the copied CSV file will be saved
copied_csv = '02_annotation_overview_scheme.csv'

# Copy the original CSV file to the new path
shutil.copyfile(original_csv, copied_csv)

print(f"File has been copied and renamed to {copied_csv}")

new_file_name = '02_annotation_overview_scheme.csv'
save_xlsx_with_date_time(new_file_name)

01_output.csv Already uploaded.

File has been copied and renamed to 02_annotation_overview_scheme.csv
File saved as: 02_annotation_overview_scheme_20250527.xlsx
